# Penalties analysis (roland_nb)

First-pass analysis for the solver penalty cap redesign. This is the working notebook;
other team members keep their own and we consolidate later.

How to read this:
- Data are SAMPLE extracts (polygon, bnb, ethereum, 2026-04-14 to 2026-06-16), one row per
  (auction_id, order_uid) winning settlement attempt. Magnitudes are illustrative; the method is
  the point.
- Analysis frame is fill-or-kill, in-market orders only (`~partially_fillable & ~is_out_of_market`).
- Native-token amounts are not comparable across chains, so we lead with cap-relative ratios, USD,
  and bps of order size. We use medians and ECDFs, not means, because the distributions are skewed.
- Only 3 chains means 3 cap levels, so cross-chain numbers are descriptive, not causal. Where we
  need a cleaner read we compare a solver against itself across the chains it operates on.

## 0. Setup and shared definitions

- load the three chains, parse the wei-string columns, apply the fok in-market filter, and
  define once the revert flag, the P&L formula, a per-chain native-USD price, and the outlier gate.
- every section below reuses these, so the definitions live in one place.

In [ ]:
import glob, os, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

# Charts render as static PNG by default (always show, no JS needed). For interactive plotly
# (zoom, hover, legend toggle), set INTERACTIVE = True and re-run, or per cell call
# fig.show(renderer="plotly_mimetype+notebook").
INTERACTIVE = False
pio.renderers.default = "plotly_mimetype+notebook" if INTERACTIVE else "png"
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 90)

WEI = 1e18
DATA_DIR = Path("../data")
NATIVE_TOKEN = {"polygon": "POL", "bnb": "BNB", "ethereum": "ETH"}
# approximate block times (seconds) for latency-in-blocks normalization; caveat only
BLOCK_SECONDS = {"polygon": 2.0, "bnb": 0.75, "ethereum": 12.0}
CHAIN_ORDER = ["polygon", "bnb", "ethereum"]
COLOR = {"polygon": "#8247E5", "bnb": "#F0B90B", "ethereum": "#627EEA"}

def wilson(k, n, z=1.96):
    # Wilson score interval for a proportion, no scipy needed
    if n == 0: return (np.nan, np.nan, np.nan)
    p = k / n; d = 1 + z*z/n
    centre = (p + z*z/(2*n)) / d
    half = (z*np.sqrt(p*(1-p)/n + z*z/(4*n*n))) / d
    return (p, centre - half, centre + half)

def winsorize(s, lo=0.01, hi=0.99):
    s = pd.to_numeric(s, errors="coerce")
    ql, qh = s.quantile(lo), s.quantile(hi)
    return s.clip(ql, qh)

def ecdf_fig(df, value, title, logx=False, frame_settled_only=False):
    fig = go.Figure()
    for ch in CHAIN_ORDER:
        v = pd.to_numeric(df.loc[df.chain == ch, value], errors="coerce").dropna()
        if logx: v = v[v > 0]
        v = np.sort(v.values)
        if len(v) == 0: continue
        y = np.arange(1, len(v)+1) / len(v)
        # downsample for plot weight
        if len(v) > 3000:
            idx = np.linspace(0, len(v)-1, 3000).astype(int); v = v[idx]; y = y[idx]
        fig.add_scatter(x=v, y=y, mode="lines", name=ch, line=dict(color=COLOR[ch]))
    fig.update_layout(title=title, yaxis_title="ECDF", xaxis_title=value)
    if logx: fig.update_xaxes(type="log")
    return fig
print("helpers ready")

In [ ]:
# Load, parse, derive. Numeric columns are stored as decimal strings of wei to keep precision.
NUM = ["volume_native","penalty_native","penalty_uncapped_native","reward_penalty_native",
       "reward_penalty_uncapped_native","reward_native","penalty_cap_native","reward_cap_upper_native",
       "order_size_usd","markout_usd","markout_relative","reference_score","observed_score",
       "slippage_native","slippage_usd","execution_cost_native","seconds_since_created",
       "seconds_to_settle","slippage_tolerance_bps","calculated_slippage_bps","block_deadline",
       "settlement_block"]
BOOL = ["settled","is_excluded_from_penalties","partially_fillable","is_out_of_market","smart_slippage"]

def to_bool(s):
    return (s.astype(str).str.strip().str.lower()
            .map({"true": True, "false": False, "1": True, "0": False}).fillna(False).astype(bool))

frames, prov = [], []
for path in sorted(glob.glob(str(DATA_DIR / "*.csv"))):
    chain = os.path.basename(path).split("_")[0]
    d = pd.read_csv(path, low_memory=False)
    d.insert(0, "chain", chain)
    for c in NUM:
        if c in d: d[c] = pd.to_numeric(d[c], errors="coerce")
    for c in BOOL:
        if c in d: d[c] = to_bool(d[c])
    frames.append(d)
    prov.append((os.path.basename(path), len(d), hashlib.md5(open(path,"rb").read()).hexdigest()[:8]))

raw = pd.concat(frames, ignore_index=True)
assert not raw["is_excluded_from_penalties"].any(), "excluded-from-penalties rows present, revisit"
print("provenance (file, rows, md5):"); [print("  ", *p) for p in prov]
print("loaded", len(raw), "rows")

In [ ]:
# fok in-market frame + derived columns
df = raw[(~raw.partially_fillable) & (~raw.is_out_of_market)].copy()
df["is_revert"] = ~df["settled"]

# native -> token
for c in ["volume_native","penalty_native","penalty_uncapped_native","reward_native",
          "reward_penalty_native","penalty_cap_native","reward_cap_upper_native",
          "slippage_native","execution_cost_native"]:
    df[c.replace("_native","_tok")] = df[c] / WEI

# per-chain native token USD price, implied from settled rows (order_size_usd / volume_tok)
sett = df[df.settled & (df.volume_tok > 0) & df.order_size_usd.notna()]
NATIVE_USD = (sett.assign(px=sett.order_size_usd / sett.volume_tok)
              .groupby("chain")["px"].median())
print("implied native-token USD price per chain:"); print(NATIVE_USD.round(4))
df["native_usd"] = df.chain.map(NATIVE_USD)

# USD size proxy available on ALL rows (incl reverts), and cap in USD
df["size_usd"] = df.volume_tok * df.native_usd
df["penalty_usd"] = df.penalty_tok * df.native_usd
df["reward_usd"] = df.reward_tok * df.native_usd
df["penalty_cap_usd"] = df.penalty_cap_tok * df.native_usd

# P&L: settled = reward_penalty + slippage - exec cost; revert = reward_penalty (= -penalty)
df["pnl_tok"] = (df.reward_penalty_tok
                 + df.slippage_tok.fillna(0.0) - df.execution_cost_tok.fillna(0.0))
df["pnl_usd"] = df.pnl_tok * df.native_usd

# cap binding + forgiven amount (reverts only meaningful)
df["cap_binds"] = df.penalty_uncapped_tok > df.penalty_cap_tok
df["forgiven_tok"] = (df.penalty_uncapped_tok - df.penalty_tok).clip(lower=0)

# penalty as bps of order size (reverts)
df["penalty_bps"] = np.where(df.size_usd > 0, df.penalty_usd / df.size_usd * 1e4, np.nan)
CAP_USD = (df.groupby("chain").penalty_cap_usd.first()).reindex(CHAIN_ORDER)
print("penalty cap in USD per chain:"); print(CAP_USD.round(2))
print("rows in fok frame:", len(df))

## 1. Data exploration

- coverage, revert balance, missingness, the ethereum native-unit outliers, and how often the
  cap binds, per chain.
- fixes the baselines and exposes the confounders (size mix, token and solver concentration)
  before any cross-chain claim.

In [ ]:
# Coverage and headline per-chain table
rows = []
for ch in CHAIN_ORDER:
    g = df[df.chain == ch]
    p, lo, hi = wilson(int(g.is_revert.sum()), len(g))
    rows.append(dict(
        chain=ch, token=NATIVE_TOKEN[ch], rows=len(g),
        orders=g.order_uid.nunique(), solvers=g.solver.nunique(),
        revert_rate=round(p,4), rr_lo=round(lo,4), rr_hi=round(hi,4),
        cap_tok=round(g.penalty_cap_tok.iloc[0],4), cap_usd=round(g.penalty_cap_usd.iloc[0],2),
        cap_binds_rate=round(g.loc[g.is_revert,"cap_binds"].mean(),4),
        med_size_usd=round(g.size_usd.median(),1),
        med_markout_rel=round(g.markout_relative.median(),5),
        med_secs_settle=round(g.seconds_to_settle.median(),1),
    ))
coverage = pd.DataFrame(rows)
display(coverage)

fig = go.Figure()
fig.add_bar(x=coverage.chain, y=coverage.revert_rate,
            error_y=dict(type="data", symmetric=False,
                         array=coverage.rr_hi-coverage.revert_rate,
                         arrayminus=coverage.revert_rate-coverage.rr_lo),
            marker_color=[COLOR[c] for c in coverage.chain])
fig.update_layout(title="Revert rate by chain (fok in-market, Wilson 95% CI)", yaxis_tickformat=".0%")
fig.show()

In [ ]:
# Missingness: settled-only columns should be null exactly on reverts
sett_only = ["order_size_usd","markout_usd","markout_relative","seconds_to_settle",
             "slippage_usd","execution_cost_native"]
miss = (df.assign(grp=np.where(df.settled,"settled","revert"))
        .groupby("grp")[sett_only].apply(lambda x: x.notna().mean()).round(3))
print("share non-null by settled/revert (settled-only columns):"); display(miss)

# Ethereum native-unit outlier diagnostic
diag = (df.groupby("chain")
        .agg(vol_p50=("volume_tok","median"), vol_p99=("volume_tok", lambda s: s.quantile(.99)),
             vol_max=("volume_tok","max"),
             unc_p99=("penalty_uncapped_tok", lambda s: s.quantile(.99)),
             unc_max=("penalty_uncapped_tok","max")).reindex(CHAIN_ORDER))
print("volume_tok / penalty_uncapped_tok tails (note ethereum max vs p99 gap):"); display(diag.round(2))

In [ ]:
# Distributions: ECDF overlays (dropdown over a few key quantities)
quants = [("size_usd","order size (USD)", True),
          ("markout_relative","markout (relative)", False),
          ("penalty_bps","penalty as bps of size (reverts)", False),
          ("seconds_to_settle","time to happy moo (s, settled)", True)]
fig = go.Figure(); buttons = []; traces_per = {}
for qi,(col,label,lx) in enumerate(quants):
    start = len(fig.data)
    sub = df if col != "penalty_bps" else df[df.is_revert]
    for ch in CHAIN_ORDER:
        v = pd.to_numeric(sub.loc[sub.chain==ch, col], errors="coerce").dropna()
        if lx: v = v[v>0]
        v = np.sort(v.values)
        if len(v)==0:
            fig.add_scatter(x=[], y=[], name=ch, visible=(qi==0)); continue
        y = np.arange(1,len(v)+1)/len(v)
        if len(v)>3000:
            idx=np.linspace(0,len(v)-1,3000).astype(int); v=v[idx]; y=y[idx]
        fig.add_scatter(x=v,y=y,mode="lines",name=ch,line=dict(color=COLOR[ch]),visible=(qi==0))
    traces_per[qi]=(start,len(fig.data))
n=len(fig.data)
for qi,(col,label,lx) in enumerate(quants):
    vis=[False]*n
    for t in range(*traces_per[qi]): vis[t]=True
    buttons.append(dict(label=label,method="update",
                        args=[{"visible":vis},{"xaxis":{"title":label,"type":"log" if lx else "linear"}}]))
fig.update_layout(title="Distributions by chain (ECDF)",
                  updatemenus=[dict(buttons=buttons,x=1.02,y=1.0)],
                  yaxis_title="ECDF", xaxis_title=quants[0][1],
                  xaxis_type="log")
fig.show()

In [ ]:
# Concentration: solver share and HHI per chain
def hhi(counts):
    s = counts/counts.sum(); return float((s*s).sum())
conc = []
for ch in CHAIN_ORDER:
    g = df[df.chain==ch]
    sc = g.solver_name.value_counts()
    conc.append(dict(chain=ch, n_solvers=g.solver.nunique(), solver_hhi=round(hhi(sc),3),
                     top_solver=sc.index[0] if len(sc) else None,
                     top_solver_share=round(sc.iloc[0]/len(g),3) if len(sc) else np.nan,
                     token_pairs=g.groupby(["sell_token","buy_token"]).ngroups))
display(pd.DataFrame(conc))

## 2. Revert rates across chains vs rewards and penalties

- per-chain revert rate next to realized rewards, realized penalties, and how often the cap binds.
- reverts are the event the cap targets; this frames whether penalties look like a deterrent or a cost solvers absorb.
- 3 chains, fully confounded by flow and competition, so read as description not cause.

In [ ]:
t1 = []
for ch in CHAIN_ORDER:
    g = df[df.chain==ch]; rev = g[g.is_revert]
    p,_,_ = wilson(int(g.is_revert.sum()), len(g))
    t1.append(dict(chain=ch, revert_rate=round(p,4),
                   med_reward_usd=round(g.loc[g.settled,"reward_usd"].median(),3),
                   med_penalty_usd=round(rev.penalty_usd.median(),3),
                   cap_usd=round(g.penalty_cap_usd.iloc[0],2),
                   cap_binds_rate=round(rev.cap_binds.mean(),3),
                   exp_penalty_per_attempt_usd=round(p*rev.penalty_usd.median(),4)))
display(pd.DataFrame(t1))

## 3. Revert rates by order size

- revert rate across order-size buckets, within each chain.
- size is the prime confounder and the lever for a variable-rate cap; we also check whether fixed penalties land hardest on small orders (penalty in bps of size).
- size uses `size_usd` (present on reverts); `order_size_usd` is settled-only.

In [ ]:
# USD size bands (comparable across chains)
bands = [0,100,1_000,10_000,100_000,np.inf]
labels = ["<$100","$100-1k","$1k-10k","$10k-100k",">$100k"]
df["size_band"] = pd.cut(df.size_usd, bands, labels=labels)
rs = []
for ch in CHAIN_ORDER:
    for b in labels:
        g = df[(df.chain==ch)&(df.size_band==b)]
        if len(g)<30: continue
        p,lo,hi = wilson(int(g.is_revert.sum()), len(g))
        rs.append(dict(chain=ch, band=b, n=len(g), revert_rate=p, lo=lo, hi=hi,
                       med_penalty_bps=g.loc[g.is_revert,"penalty_bps"].median()))
rs = pd.DataFrame(rs)
fig = px.line(rs, x="band", y="revert_rate", color="chain", markers=True,
              color_discrete_map=COLOR, category_orders={"band":labels},
              title="Revert rate by order-size band")
fig.update_yaxes(tickformat=".0%"); fig.show()

fig2 = px.line(rs, x="band", y="med_penalty_bps", color="chain", markers=True,
               color_discrete_map=COLOR, category_orders={"band":labels},
               title="Median penalty as bps of order size, by size band (reverts)")
fig2.show()

## 4. Does a higher penalty cap go with lower rewards and lower solver P&L?

- per-chain rewards and P&L vs cap, plus the cleaner within-solver comparison (solvers active on more than one chain, compared against themselves).
- the core economic question, and P&L is the discriminator for the ambiguous null (flat markout + negative P&L means solvers eat the cap; reward and penalty caps are asymmetric).

In [ ]:
# P&L formula reminder: settled = reward_penalty + slippage - exec_cost ; revert = -penalty.
t3 = (df.groupby("chain")
      .agg(med_reward_usd=("reward_usd", lambda s: s[df.loc[s.index,"settled"]].median()),
           med_pnl_usd=("pnl_usd","median"),
           neg_pnl_share=("pnl_usd", lambda s: (s<0).mean()),
           cap_usd=("penalty_cap_usd","first")).reindex(CHAIN_ORDER).round(4))
display(t3)

# within-solver: per (solver, chain) median pnl, for solvers on >=2 chains with >=100 rows each
ps = (df.groupby(["solver","chain"])
      .agg(n=("pnl_usd","size"), med_pnl=("pnl_usd","median"),
           med_reward=("reward_usd", lambda s: s.median())).reset_index())
ps = ps[ps.n>=100]
multi = ps.groupby("solver").chain.nunique()
multi = multi[multi>=2].index
pm = ps[ps.solver.isin(multi)].copy()
cap_rank = {c:r for r,c in enumerate(CAP_USD.sort_values().index)}
pm["cap_rank"] = pm.chain.map(cap_rank)
print(f"{len(multi)} solvers active on >=2 chains (>=100 rows/chain)")
# paired: each solver's pnl on its highest-cap vs lowest-cap chain
pairs = []
for s,g in pm.groupby("solver"):
    g = g.sort_values("cap_rank")
    lo, hi = g.iloc[0], g.iloc[-1]
    pairs.append(dict(solver=s, low_chain=lo.chain, high_chain=hi.chain,
                      pnl_low=lo.med_pnl, pnl_high=hi.med_pnl, diff=hi.med_pnl-lo.med_pnl))
pairs = pd.DataFrame(pairs)
if len(pairs):
    print("median within-solver P&L diff (high-cap minus low-cap chain):", round(pairs["diff"].median(),4))
    print("share of solvers with lower P&L on their higher-cap chain:", round((pairs["diff"]<0).mean(),3))
    fig = go.Figure()
    for _,r in pairs.iterrows():
        fig.add_scatter(x=[r.pnl_low, r.pnl_high], y=[r.solver, r.solver], mode="lines+markers",
                        line=dict(color="lightgray"), showlegend=False)
    fig.update_layout(title="Within-solver median P&L (USD): low-cap (left) vs high-cap (right) chain",
                      xaxis_title="median P&L per attempt (USD)")
    fig.show()
display(pairs.round(4))

## 5. Is a higher cap priced into worse bids (worse markout)?

- markout per chain, and within-solver markout on the higher-cap vs lower-cap chain.
- tests whether cap risk is passed to users as worse execution.
- markout is settled-only, so strategic reverting of bad trades can make survivors look better (survivorship); read jointly with Section 2.

In [ ]:
fig = ecdf_fig(df[df.settled], "markout_relative",
               "Markout (relative) by chain, settled rows", logx=False)
fig.show()

# within-solver markout (same multi-chain solver set)
psm = (df[df.settled].groupby(["solver","chain"]).markout_relative.median().reset_index())
psm = psm[psm.solver.isin(multi)].copy(); psm["cap_rank"]=psm.chain.map(cap_rank)
mp = []
for s,g in psm.groupby("solver"):
    g=g.sort_values("cap_rank")
    if g.chain.nunique()<2: continue
    mp.append(dict(solver=s, mk_low=g.iloc[0].markout_relative, mk_high=g.iloc[-1].markout_relative,
                   diff=g.iloc[-1].markout_relative-g.iloc[0].markout_relative))
mp=pd.DataFrame(mp)
if len(mp):
    print("median within-solver markout diff (high-cap minus low-cap):", round(mp["diff"].median(),6))
    print("share with worse (lower) markout on higher-cap chain:", round((mp["diff"]<0).mean(),3))
display(mp.round(6))

## 6. Three outcomes vs cap level: profit, markout, time to happy moo

- solver P&L, markout, and settle latency lined up against each chain's cap.
- the redesign needs the whole trade-off; hypothesis is higher caps go with worse markout and slower settlement.
- latency is shown in seconds and in blocks (chains have very different block times).

In [ ]:
o = (df[df.settled].groupby("chain")
     .agg(med_pnl_usd=("pnl_usd","median"),
          med_markout=("markout_relative","median"),
          med_secs=("seconds_to_settle","median")).reindex(CHAIN_ORDER))
o["med_blocks"] = o.med_secs / pd.Series(BLOCK_SECONDS).reindex(o.index)
o["cap_usd"] = CAP_USD
o = o.reset_index().sort_values("cap_usd")
display(o.round(4))

metrics = [("med_pnl_usd","median P&L (USD)"),("med_markout","median markout (rel)"),
           ("med_secs","median time to happy moo (s)"),("med_blocks","median blocks to settle")]
fig = go.Figure(); buttons=[]
for i,(col,label) in enumerate(metrics):
    fig.add_bar(x=o.chain, y=o[col], name=label, visible=(i==0),
                marker_color=[COLOR[c] for c in o.chain])
    vis=[j==i for j in range(len(metrics))]
    buttons.append(dict(label=label,method="update",args=[{"visible":vis},{"yaxis":{"title":label}}]))
fig.update_layout(title="Outcome by chain (ordered low to high cap)",
                  updatemenus=[dict(buttons=buttons,x=1.02,y=1.0)], yaxis_title=metrics[0][1])
fig.show()

## 7. Distribution of capped vs uncapped penalties, per chain

- how often the cap binds and how much penalty it forgives.
- sizes the problem; if the cap rarely binds it is a non-event, if it clips heavily the redesign matters.
- uncapped penalty is the most ethereum-outlier-exposed field, so totals are winsorized and the raw is shown beside.

In [ ]:
rev = df[df.is_revert].copy()
t6 = []
for ch in CHAIN_ORDER:
    g = rev[rev.chain==ch]
    forg_raw = g.forgiven_tok.sum()
    forg_w = winsorize(g.forgiven_tok).sum()
    t6.append(dict(chain=ch, reverts=len(g), cap_binds_rate=round(g.cap_binds.mean(),3),
                   total_penalty_tok=round(g.penalty_tok.sum(),2),
                   forgiven_tok_winsor=round(forg_w,2), forgiven_tok_raw=round(forg_raw,2)))
display(pd.DataFrame(t6))

# ECDF of uncapped penalty relative to cap (log x), vertical line at 1 = cap
rev["unc_over_cap"] = rev.penalty_uncapped_tok / rev.penalty_cap_tok
fig = ecdf_fig(rev, "unc_over_cap", "Uncapped penalty / cap, by chain (reverts)", logx=True)
fig.add_vline(x=1.0, line_dash="dash", line_color="gray")
fig.show()

## 8. Counterfactual: variable-rate penalty (% of order size)

- recompute penalties if the cap were a flat rate on order size, holding the observed reverts fixed, and find the revenue-neutral rate per chain.
- the leading redesign candidate; shows who pays more or less and how penalty mass shifts by size.
- assumes solvers do not change behavior (so it understates large-order relief), and size uses the USD proxy with outlier gating.

In [ ]:
# size gate: drop implausible USD sizes for the counterfactual (ethereum scaling artifacts)
gate = df[df.is_revert & df.size_usd.notna() & (df.size_usd < df.size_usd.quantile(0.999))].copy()
rates = [1,2,5,10,25,50]
rowsr = []
for ch in CHAIN_ORDER:
    g = gate[gate.chain==ch]; realized = g.penalty_usd.sum()
    for r in rates:
        var = (r/1e4 * g.size_usd).sum()
        rowsr.append(dict(chain=ch, rate_bps=r, variable_usd=var, realized_usd=realized,
                          ratio=var/realized if realized else np.nan))
cf = pd.DataFrame(rowsr)
fig = px.line(cf, x="rate_bps", y="ratio", color="chain", markers=True, color_discrete_map=COLOR,
              title="Variable-rate total vs realized fixed-cap (1.0 = revenue neutral)")
fig.add_hline(y=1.0, line_dash="dash", line_color="gray")
fig.update_xaxes(title="penalty rate (bps of order size)"); fig.show()

# revenue-neutral rate per chain (interp on the realized total)
rn = []
for ch in CHAIN_ORDER:
    g = gate[gate.chain==ch]; realized = g.penalty_usd.sum(); base = g.size_usd.sum()
    rstar = realized / base * 1e4 if base else np.nan   # r* solving r*base = realized
    rn.append(dict(chain=ch, revenue_neutral_bps=round(rstar,2),
                   realized_usd=round(realized,1), reverts=len(g)))
display(pd.DataFrame(rn))

## 9. Counterfactual: non-economic penalty (sit out X blocks)

- approximate suspending a reverting solver for X blocks; estimate rewards lost, penalties avoided, and order coverage at risk.
- a non-economic alternative to a monetary cap.
- we only see winning attempts (no full bid book), so we reassign only to an observed alternative settler of the same order; everything else is reported as coverage-at-risk, the honesty metric.

In [ ]:
# Model A: suspend solver for X blocks after a revert; remove its subsequent winning attempts.
# An order keeps coverage only if another (non-suspended) solver actually settled it later.
order_settler = (df[df.settled].sort_values("block_deadline")
                 .groupby("order_uid").agg(settle_block=("block_deadline","first"),
                                            settle_solver=("solver","first")))
def sim_sitout(g, X):
    g = g.sort_values("block_deadline")
    suspended_until = {}
    lost_reward = 0.0; avoided_penalty = 0.0; removed = 0; at_risk = 0
    for r in g.itertuples():
        b = r.block_deadline
        if suspended_until.get(r.solver, -1) >= b:
            removed += 1
            if r.settled:
                lost_reward += r.reward_usd if not np.isnan(r.reward_usd) else 0.0
                # coverage: did another solver settle this order later?
                os_ = order_settler.loc[r.order_uid] if r.order_uid in order_settler.index else None
                if os_ is None or os_.settle_solver == r.solver:
                    at_risk += 1
            else:
                avoided_penalty += r.penalty_usd if not np.isnan(r.penalty_usd) else 0.0
            continue
        if r.is_revert:
            suspended_until[r.solver] = b + X
    return dict(removed=removed, lost_reward_usd=lost_reward,
                avoided_penalty_usd=avoided_penalty, coverage_at_risk=at_risk)

res = []
for X in [1,5,25,100]:
    for ch in CHAIN_ORDER:
        d = sim_sitout(df[df.chain==ch], X); d.update(chain=ch, X_blocks=X); res.append(d)
sit = pd.DataFrame(res)[["chain","X_blocks","removed","lost_reward_usd","avoided_penalty_usd","coverage_at_risk"]]
display(sit.round(1))
fig = px.line(sit, x="X_blocks", y="coverage_at_risk", color="chain", markers=True,
              color_discrete_map=COLOR, title="Orders at coverage risk vs sit-out length")
fig.update_xaxes(type="log"); fig.show()